In [1]:
# =========================================================
# DATAHUB SETUP & IMPORTS. (May take a while!) Comment me out after first run!
# =========================================================

# remove old kernel
!jupyter kernelspec uninstall -y cse151b

# Remove old .venv
!rm -rf .venv

print("Old environment and kernel destroyed.")

# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

# Create a virtual environment
!~/.local/bin/uv venv .venv --seed

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install sympy numpy transformers dspy vllm==0.19.1 tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!")

Couldn't find kernel spec(s): cse151b


rm: cannot remove '.venv/lib/python3.13/site-packages/tornado/.nfs000000000140417300000c9e': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/psutil/.nfs000000000140601500000c9f': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/pyzmq.libs/.nfs000000000140543900000ca0': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/pyzmq.libs/.nfs000000000140543800000ca1': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/zmq/backend/cython/.nfs000000000140546100000ca2': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_sys_monitoring/.nfs00000000009a9f0d00000ca3': Device or resource busy
rm: cannot remove '.venv/lib/python3.13/site-packages/debugpy/_vendored/pydevd/_pydevd_bundle/.nfs00000000009a9eb100000ca4': Device or resource busy


downloading uv 0.11.17 x86_64-unknown-linux-gnu


installing to /home/a5verma/.local/bin


  uv
  uvx
everything's installed!


...

Old environment and kernel destroyed.


Using CPython 3.13.13 interpreter at: /opt/conda/bin/python
Creating virtual environment with seed packages at: .venv
error: Failed to create virtual environment
  Caused by: A directory already exists at: .venv

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to replace the existing directory


/bin/bash: line 1: .venv/bin/python: No such file or directory


/bin/bash: line 1: .venv/bin/python: No such file or directory


Done. Restart the kernel before proceeding. Then click on current kernel -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'. Restart kernel again. Comment out this code!


In [30]:
import dspy
from dspy.teleprompt import GEPA
import json
import os
import re
import sys
from pathlib import Path
from typing import Optional

STRATIFIED_DATA   = "data/16k_max_openr1_math_7_5k_stratified.jsonl"
PUBLIC_DATA       = "data/public.jsonl"
PRIVATE_DATA      = "data/private.jsonl"

In [31]:
# Cell 2: Core DSPy Definitions
import dspy
import re

# 1. Align the DSPy Signature directly with your ('prompt', 'completion') format
class MathReasoning(dspy.Signature):
    """Solve the math problem step-by-step with explicit logical deductions."""
    prompt = dspy.InputField(desc="The mathematical prompt or question.")
    completion = dspy.OutputField(desc="The step-by-step reasoning followed by the final answer inside \\boxed{}.")

# 2. Wrap signature into a ChainOfThought Module
class MathSolver(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(MathReasoning)
        
    def forward(self, prompt):
        return self.prog(prompt=prompt)

# Helper function to extract answers from the completion text block
def extract_answer(text):
    match = re.search(r'\\boxed\{(.*?)\}', str(text))
    if match:
        return match.group(1).strip().lower()
    return str(text).strip().lower()

# 3. Rich Textual Feedback Metric for GEPA Evolutionary Proposer
def math_feedback_metric(gold, pred, trace=None, pred_name=None, pred_trace=None):
    target_ans = extract_answer(example.completion)
    predicted_ans = extract_answer(pred.completion)
    
    correct = target_ans == predicted_ans
    score = 1.0 if correct else 0.0
    
    feedback = "Perfect solution." if correct else (
        f"Error found. Target answer expected: {target_ans}, but model predicted: {predicted_ans}.\n"
        f"Model completion output trace: {pred.completion}\n"
        f"Identify the mathematical or step deduction breakdown and update prompt instructions to avoid it."
    )
    return dspy.Prediction(score=score, feedback=feedback)

In [34]:
# Cell 3: Stage 1 Public Optimization & Data Ingestion
from dspy.teleprompt import GEPA

print("=== Loading Data in ('prompt', 'completion') Format ===")

my_raw_public_data = [[json.loads(line) for line in open(STRATIFIED_DATA)][0]] # Use only the first item to test
public_trainset = [
    dspy.Example(prompt=item["prompt"], completion=item["completion"]).with_inputs("prompt")
    for item in my_raw_public_data
]

print(f"Successfully ingested {len(public_trainset)} training instances.")
print("\n=== Running Stage 1: Public Baseline Prompt Bootstrapping ===")

# 1. Define a strong frontier model specifically for text prompt reflection
reflection_model_stage1 = dspy.LM('openai/gpt-4o', temperature=1.0, max_tokens=4096)

# 2. CRITICAL FIX: Explicitly assign the reflection model using the reflection_lm parameter
optimizer_stage1 = GEPA(
    metric=math_feedback_metric,
    reflection_lm=reflection_model_stage1,
    auto='light'
)

public_bootstrapped_solver = optimizer_stage1.compile(
    student=MathSolver(),
    trainset=public_trainset
)

# Persist baseline configurations to disk
public_bootstrapped_solver.save("public_baseline_prompts.json")
print("Stage 1 execution complete! Baseline prompts exported to disk.")

2026/05/31 17:52:45 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 388 metric calls of the program. This amounts to 194.00 full evals on the train set.


2026/05/31 17:52:45 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.


2026/05/31 17:52:45 INFO dspy.teleprompt.gepa.gepa: Using 2 examples for tracking Pareto scores.


=== Loading Data in ('prompt', 'completion') Format ===
Successfully ingested 2 training instances.

=== Running Stage 1: Public Baseline Prompt Bootstrapping ===


GEPA Optimization:   0%|          | 0/388 [00:00<?, ?rollouts/s]

...

2026/05/31 18:15:06 WARNING dspy.teleprompt.bootstrap_trace: Failed to unpack prediction and trace. This is likely due to the LLM response not following dspy formatting.


2026/05/31 18:15:06 INFO dspy.teleprompt.gepa.gepa: Iteration 129: No trajectories captured. Skipping.


2026/05/31 18:15:06 INFO dspy.teleprompt.gepa.gepa: Iteration 129: Reflective mutation did not propose a new candidate


GEPA Optimization:  99%|█████████▉| 386/388 [22:21<00:06,  3.47s/rollouts]


Stage 1 execution complete! Baseline prompts exported to disk.


In [ ]:
# Cell 4: Stage 2 Private Optimization
from dspy.teleprompt import GEPA

print("=== Loading Private Data & Running Adaptation ===")

my_raw_private_data = [json.loads(line) for line in open(PRIVATE_PATH)]

private_trainset = [
    dspy.Example(prompt=item["prompt"], completion=item["completion"]).with_inputs("prompt")
    for item in my_raw_private_data
]

# 1. Warm start a fresh module with the public prompt instructions
warm_started_solver = MathSolver()
warm_started_solver.load("public_baseline_prompts.json")

# 2. Initialize local Qwen3-4B-Thinking-2507 model
private_lm = dspy.LM(
    "huggingface/Qwen/Qwen3-4B-Thinking-2507", 
    temperature=0.6, 
    max_tokens=16384, # Crucial large token ceiling to allow Qwen's long native <think> trace loops
    device_map="auto"
)
dspy.settings.configure(lm=private_lm)

# 3. Optimize the prompt instructions strictly for the private dataset
optimizer_stage2 = GEPA(metric=math_feedback_metric)
final_private_solver = optimizer_stage2.compile(
    student=warm_started_solver,
    trainset=private_trainset
)

# 4. Save final combined prompt profiles
final_private_solver.save("private_final_prompts.json")
print("Stage 2 execution complete! Customized prompts saved to private_final_prompts.json.")

In [ ]:
# Cell 5: Independent Evaluation Matrix
from dspy.evaluate import Evaluate

print("=== Running Stage 3: Performance Validation on Unseen Test Split ===")

# Separate mock partition simulating your unseen private evaluation test split
my_unseen_test_data = [
    {
        "prompt": "Evaluate systemic cryptographic variance given a token key of 102...", 
        "completion": "Computing the variance factor over the sequence maps directly to 102. The answer is \\boxed{102}."
    }
]

private_testset = [
    dspy.Example(prompt=item["prompt"], completion=item["completion"]).with_inputs("prompt")
    for item in my_unseen_test_data
]

# Apply strict production inference sampling metrics mandated by Alibaba
production_model = dspy.LM(
    "huggingface/Qwen/Qwen3-4B-Thinking-2507",
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    max_tokens=16384,
    device_map="auto"
)
dspy.settings.configure(lm=production_model)

# Mount the finalized optimized prompt rules payload
production_solver = MathSolver()
production_solver.load("private_final_prompts.json")

# Evaluate across validation data that the pipeline has never encountered
evaluator = Evaluate(
    devset=private_testset,
    metric=math_feedback_metric,
    num_threads=2, # Scale based on host system VRAM concurrency boundaries
    display_progress=True,
    display_table=True
)

eval_result = evaluator(production_solver)
print(f"\n" + "="*55)
print(f"Final Accuracy of Qwen3-4B-Thinking-2507 on Unseen Private Test Data: {eval_result.score * 100:.2%}")
print("="*55)

In [2]:
SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices carefully. "
    "Think through the problem step by step, then select the single best answer. "
    "At the very end of your response, output the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels    = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question